<a href="https://colab.research.google.com/github/Sprg72/Data-Engineer/blob/main/notebooks/Pyspark_lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# step1
!pip install findspark pyspark


In [2]:
# step2
import findspark
findspark.init()

In [5]:
#step3
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('myapp').getOrCreate()

In [6]:
sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=myapp>

In [7]:
# joins.
# to collect data from two RDDs based on Key column.
# inner joins ---> matching elements based on key
# left outer joins --> matching elements and non matching elements of left side
# right outer joins --> matching elements and non matching elements of right side
# full outer joins ---> matching elements and non matching elements of both sides
# cross joins.  --> cartesian product --> each element of left side joins with each element of right side.


In [8]:

# Note: RDD1.join(RDD2) ---> joins expecting data in key value pair tuples.

In [9]:
data1 = [(101, 'Amar'), (102, 'Amala'), (103, 'Ankit'), (104, 'Anusha')]
R1 = sc.parallelize(data1)
data2 = [(101, 'Hyderabad'), (102, 'Delhi'), (105, 'Pune'), (106, 'Hyderabad')]
R2 = sc.parallelize(data2)

In [10]:
# inner join:
ij = R1.join(R2)
ij.collect()

[(101, ('Amar', 'Hyderabad')), (102, ('Amala', 'Delhi'))]

In [11]:
ijinfo = ij.map(lambda x: (x[0], x[1][0], x[1][1]))
ijinfo.collect()

[(101, 'Amar', 'Hyderabad'), (102, 'Amala', 'Delhi')]

In [12]:
# left outer join
loj = R1.leftOuterJoin(R2)
loj.collect()

[(104, ('Anusha', None)),
 (101, ('Amar', 'Hyderabad')),
 (102, ('Amala', 'Delhi')),
 (103, ('Ankit', None))]

In [13]:
# right outer join
roj = R1.rightOuterJoin(R2)
roj.collect()

[(101, ('Amar', 'Hyderabad')),
 (105, (None, 'Pune')),
 (102, ('Amala', 'Delhi')),
 (106, (None, 'Hyderabad'))]

In [14]:
# full outer join
foj = R1.fullOuterJoin(R2)
foj.collect()

[(104, ('Anusha', None)),
 (101, ('Amar', 'Hyderabad')),
 (105, (None, 'Pune')),
 (102, ('Amala', 'Delhi')),
 (106, (None, 'Hyderabad')),
 (103, ('Ankit', None))]

In [15]:
# cross joins
cr = R1.cartesian(R2)
cr.count()

16

In [16]:
cr.collect()

[((101, 'Amar'), (101, 'Hyderabad')),
 ((101, 'Amar'), (102, 'Delhi')),
 ((102, 'Amala'), (101, 'Hyderabad')),
 ((102, 'Amala'), (102, 'Delhi')),
 ((101, 'Amar'), (105, 'Pune')),
 ((101, 'Amar'), (106, 'Hyderabad')),
 ((102, 'Amala'), (105, 'Pune')),
 ((102, 'Amala'), (106, 'Hyderabad')),
 ((103, 'Ankit'), (101, 'Hyderabad')),
 ((103, 'Ankit'), (102, 'Delhi')),
 ((104, 'Anusha'), (101, 'Hyderabad')),
 ((104, 'Anusha'), (102, 'Delhi')),
 ((103, 'Ankit'), (105, 'Pune')),
 ((103, 'Ankit'), (106, 'Hyderabad')),
 ((104, 'Anusha'), (105, 'Pune')),
 ((104, 'Anusha'), (106, 'Hyderabad'))]



```
# input file1 : emp1.txt
101, amar,90000,m,11
102, amala,20000,f,12
103, ankit,40000,m,13
104, ankita,60000,f,13
105, anusha,110000,f,12
106, anuz,20000,m,11
107, akash,100000,m,12
108, siva,20000,m,14
109, sivani,30000,f,15
110, mani,30000,m,12
111, manisha,300000,f,13
112, sivam,200000,m,12
113, varun,200000,m,13


input file2: dept.txt
11,marketing,hyd,m1
12,hr,delhi,m2
13,Finance,hyd,m1
20,admin,delhi,m2
21,IT,hyd,m1

expected output schema:
   id,name,salary,gender,dno,dname,location,mid
   110,mani,300000,m,12,Hr,delhi,m2



data1 = sc.textFile('/content/emp1.txt')
data2 = sc.textFile('/content/dept.txt')



('11', (['106', ' anuz', '20000', 'm'], ['marketing', 'hyd', 'm1']))
('14', (['108', ' siva', '20000', 'm'], None))
('20', (None, ['admin', 'delhi', 'm2']))


def cleanjoin(x): #x is a tuple
  dno = x[0]
  linfo = x[1][0]
  rinfo = x[1][1]
  if linfo == None:
     linfo = ['NA'] * 4
  elif rinfo == None:
       rinfo = ['NA'] * 3
  info = linfo + [dno] + rinfo
  outline = ','.join(info)
  return outline


```



In [17]:
data1 = sc.textFile('/content/emp1.txt')
data2 = sc.textFile('/content/dept.txt')

In [18]:
data1.collect()

['101, amar,90000,m,11',
 '102, amala,20000,f,12',
 '103, ankit,40000,m,13',
 '104, ankita,60000,f,13',
 '105, anusha,110000,f,12',
 '106, anuz,20000,m,11',
 '107, akash,100000,m,12',
 '108, siva,20000,m,14',
 '109, sivani,30000,f,15',
 '110, mani,30000,m,12',
 '111, manisha,300000,f,13',
 '112, sivam,200000,m,12',
 '113, varun,200000,m,13']

In [24]:
words1= data1.map(lambda x : x.split(','))
emp = words1.map(lambda x : (x[-1], x[:-1]))
emp.collect()


[('11', ['101', ' amar', '90000', 'm']),
 ('12', ['102', ' amala', '20000', 'f']),
 ('13', ['103', ' ankit', '40000', 'm']),
 ('13', ['104', ' ankita', '60000', 'f']),
 ('12', ['105', ' anusha', '110000', 'f']),
 ('11', ['106', ' anuz', '20000', 'm']),
 ('12', ['107', ' akash', '100000', 'm']),
 ('14', ['108', ' siva', '20000', 'm']),
 ('15', ['109', ' sivani', '30000', 'f']),
 ('12', ['110', ' mani', '30000', 'm']),
 ('13', ['111', ' manisha', '300000', 'f']),
 ('12', ['112', ' sivam', '200000', 'm']),
 ('13', ['113', ' varun', '200000', 'm'])]

In [20]:
data2.collect()

['11,marketing,hyd,m1',
 '12,hr,delhi,m2',
 '13,Finance,hyd,m1',
 '20,admin,delhi,m2',
 '21,IT,hyd,m1']

In [25]:
words2 = data2.map(lambda x : x.split(','))
words2.collect()

[['11', 'marketing', 'hyd', 'm1'],
 ['12', 'hr', 'delhi', 'm2'],
 ['13', 'Finance', 'hyd', 'm1'],
 ['20', 'admin', 'delhi', 'm2'],
 ['21', 'IT', 'hyd', 'm1']]

In [26]:
dept = words2.map(lambda x : (x[0], x[1:]))
dept.collect()

[('11', ['marketing', 'hyd', 'm1']),
 ('12', ['hr', 'delhi', 'm2']),
 ('13', ['Finance', 'hyd', 'm1']),
 ('20', ['admin', 'delhi', 'm2']),
 ('21', ['IT', 'hyd', 'm1'])]

In [27]:
# inner join
ediner = emp.join(dept)
ediner.collect()

[('12', (['102', ' amala', '20000', 'f'], ['hr', 'delhi', 'm2'])),
 ('12', (['105', ' anusha', '110000', 'f'], ['hr', 'delhi', 'm2'])),
 ('12', (['107', ' akash', '100000', 'm'], ['hr', 'delhi', 'm2'])),
 ('12', (['110', ' mani', '30000', 'm'], ['hr', 'delhi', 'm2'])),
 ('12', (['112', ' sivam', '200000', 'm'], ['hr', 'delhi', 'm2'])),
 ('13', (['103', ' ankit', '40000', 'm'], ['Finance', 'hyd', 'm1'])),
 ('13', (['104', ' ankita', '60000', 'f'], ['Finance', 'hyd', 'm1'])),
 ('13', (['111', ' manisha', '300000', 'f'], ['Finance', 'hyd', 'm1'])),
 ('13', (['113', ' varun', '200000', 'm'], ['Finance', 'hyd', 'm1'])),
 ('11', (['101', ' amar', '90000', 'm'], ['marketing', 'hyd', 'm1'])),
 ('11', (['106', ' anuz', '20000', 'm'], ['marketing', 'hyd', 'm1']))]

In [29]:
# left out join
edleft = emp.leftOuterJoin(dept)
edleft.collect()

[('12', (['102', ' amala', '20000', 'f'], ['hr', 'delhi', 'm2'])),
 ('12', (['105', ' anusha', '110000', 'f'], ['hr', 'delhi', 'm2'])),
 ('12', (['107', ' akash', '100000', 'm'], ['hr', 'delhi', 'm2'])),
 ('12', (['110', ' mani', '30000', 'm'], ['hr', 'delhi', 'm2'])),
 ('12', (['112', ' sivam', '200000', 'm'], ['hr', 'delhi', 'm2'])),
 ('13', (['103', ' ankit', '40000', 'm'], ['Finance', 'hyd', 'm1'])),
 ('13', (['104', ' ankita', '60000', 'f'], ['Finance', 'hyd', 'm1'])),
 ('13', (['111', ' manisha', '300000', 'f'], ['Finance', 'hyd', 'm1'])),
 ('13', (['113', ' varun', '200000', 'm'], ['Finance', 'hyd', 'm1'])),
 ('14', (['108', ' siva', '20000', 'm'], None)),
 ('15', (['109', ' sivani', '30000', 'f'], None)),
 ('11', (['101', ' amar', '90000', 'm'], ['marketing', 'hyd', 'm1'])),
 ('11', (['106', ' anuz', '20000', 'm'], ['marketing', 'hyd', 'm1']))]

In [31]:
# right outer join
edright = emp.rightOuterJoin(dept)
edright.collect()

[('21', (None, ['IT', 'hyd', 'm1'])),
 ('12', (['102', ' amala', '20000', 'f'], ['hr', 'delhi', 'm2'])),
 ('12', (['105', ' anusha', '110000', 'f'], ['hr', 'delhi', 'm2'])),
 ('12', (['107', ' akash', '100000', 'm'], ['hr', 'delhi', 'm2'])),
 ('12', (['110', ' mani', '30000', 'm'], ['hr', 'delhi', 'm2'])),
 ('12', (['112', ' sivam', '200000', 'm'], ['hr', 'delhi', 'm2'])),
 ('20', (None, ['admin', 'delhi', 'm2'])),
 ('13', (['103', ' ankit', '40000', 'm'], ['Finance', 'hyd', 'm1'])),
 ('13', (['104', ' ankita', '60000', 'f'], ['Finance', 'hyd', 'm1'])),
 ('13', (['111', ' manisha', '300000', 'f'], ['Finance', 'hyd', 'm1'])),
 ('13', (['113', ' varun', '200000', 'm'], ['Finance', 'hyd', 'm1'])),
 ('11', (['101', ' amar', '90000', 'm'], ['marketing', 'hyd', 'm1'])),
 ('11', (['106', ' anuz', '20000', 'm'], ['marketing', 'hyd', 'm1']))]

In [32]:
# full outer join
edfull = emp.fullOuterJoin(dept)
edfull.collect()

[('21', (None, ['IT', 'hyd', 'm1'])),
 ('12', (['102', ' amala', '20000', 'f'], ['hr', 'delhi', 'm2'])),
 ('12', (['105', ' anusha', '110000', 'f'], ['hr', 'delhi', 'm2'])),
 ('12', (['107', ' akash', '100000', 'm'], ['hr', 'delhi', 'm2'])),
 ('12', (['110', ' mani', '30000', 'm'], ['hr', 'delhi', 'm2'])),
 ('12', (['112', ' sivam', '200000', 'm'], ['hr', 'delhi', 'm2'])),
 ('20', (None, ['admin', 'delhi', 'm2'])),
 ('13', (['103', ' ankit', '40000', 'm'], ['Finance', 'hyd', 'm1'])),
 ('13', (['104', ' ankita', '60000', 'f'], ['Finance', 'hyd', 'm1'])),
 ('13', (['111', ' manisha', '300000', 'f'], ['Finance', 'hyd', 'm1'])),
 ('13', (['113', ' varun', '200000', 'm'], ['Finance', 'hyd', 'm1'])),
 ('14', (['108', ' siva', '20000', 'm'], None)),
 ('15', (['109', ' sivani', '30000', 'f'], None)),
 ('11', (['101', ' amar', '90000', 'm'], ['marketing', 'hyd', 'm1'])),
 ('11', (['106', ' anuz', '20000', 'm'], ['marketing', 'hyd', 'm1']))]

In [33]:
# expected output:
# 101,amar,90000,m,11,marketing,hyd,m1
# NA,NA,NA,NA,NA,20,admin,delhi,m2

# 108,siva,200000,m14,NA,NA,NA

In [43]:
def cleanjoin(x): #x is a tuple
  dno = x[0]
  linfo = x[1][0]
  rinfo = x[1][1]
  if linfo == None:
     linfo = ['NA'] * 4
  elif rinfo == None:
       rinfo = ['NA'] * 3
  info = linfo + [dno] + rinfo
  outline = ','.join(info)
  return outline

In [44]:
cleanjoin(('11', (['106', ' anuz', '20000', 'm'], ['marketing', 'hyd', 'm1'])))

'106, anuz,20000,m,11,marketing,hyd,m1'

In [45]:
cleanjoin(('14', (['108', ' siva', '20000', 'm'], None)))

'108, siva,20000,m,14,NA,NA,NA'

In [46]:
cleanjoin(('20', (None, ['admin', 'delhi', 'm2'])))

'NA,NA,NA,NA,20,admin,delhi,m2'

In [47]:
fullinfo = edfull.map(cleanjoin)
fullinfo.collect()

['NA,NA,NA,NA,21,IT,hyd,m1',
 '102, amala,20000,f,12,hr,delhi,m2',
 '105, anusha,110000,f,12,hr,delhi,m2',
 '107, akash,100000,m,12,hr,delhi,m2',
 '110, mani,30000,m,12,hr,delhi,m2',
 '112, sivam,200000,m,12,hr,delhi,m2',
 'NA,NA,NA,NA,20,admin,delhi,m2',
 '103, ankit,40000,m,13,Finance,hyd,m1',
 '104, ankita,60000,f,13,Finance,hyd,m1',
 '111, manisha,300000,f,13,Finance,hyd,m1',
 '113, varun,200000,m,13,Finance,hyd,m1',
 '108, siva,20000,m,14,NA,NA,NA',
 '109, sivani,30000,f,15,NA,NA,NA',
 '101, amar,90000,m,11,marketing,hyd,m1',
 '106, anuz,20000,m,11,marketing,hyd,m1']

In [48]:
fullinfo.coalesce(1).saveAsTextFile('/content/efull')



```
# output file content part-00000:

NA,NA,NA,NA,21,IT,hyd,m1
102, amala,20000,f,12,hr,delhi,m2
105, anusha,110000,f,12,hr,delhi,m2
107, akash,100000,m,12,hr,delhi,m2
110, mani,30000,m,12,hr,delhi,m2
112, sivam,200000,m,12,hr,delhi,m2
NA,NA,NA,NA,20,admin,delhi,m2
103, ankit,40000,m,13,Finance,hyd,m1
104, ankita,60000,f,13,Finance,hyd,m1
111, manisha,300000,f,13,Finance,hyd,m1
113, varun,200000,m,13,Finance,hyd,m1
108, siva,20000,m,14,NA,NA,NA
109, sivani,30000,f,15,NA,NA,NA
101, amar,90000,m,11,marketing,hyd,m1
106, anuz,20000,m,11,marketing,hyd,m1

```

